# Semana 3 — Una función de búsqueda, cuatro algoritmos

**Plan de la lección:** [`../weeks/week-03.md`](../weeks/week-03.md) · **Diapositivas:** [`../slides/week-03/deck.md`](../slides/week-03/deck.md)

Programación en vivo, 25 minutos — los 25 minutos más valiosos de la semana, porque
convierte cuatro algoritmos en **una sola idea más una disciplina de frontera**.

> ⚠️ **La función `search()` que escribas aquí se reutiliza en la semana 4 (A\* es una diferencia de 10 líneas)
> y en la semana 9 (el planificador hacia adelante).** Si está rota, seguirá rota hasta la semana 9.
> Corrígela ahora.

## 1 · La interfaz `Problem`

Todo lo que se trabaja entre las semanas 3 y 9 utiliza estos cinco métodos. Esta es la
*formulación del problema en cinco componentes* de las diapositivas, llevada a código.

In [1]:
# Importa deque, una estructura de cola eficiente.
from collections import deque

# Importa heapq, que permite trabajar con colas de prioridad.
import heapq

# Importa itertools, útil para generar secuencias y contadores.
import itertools


# Clase base que representa un problema de búsqueda.
# Define los elementos necesarios para describir el problema.
class Problem:
    """The five components: initial state, actions, transition, goal test, cost."""

    # Constructor de la clase.
    # initial representa el estado inicial.
    # goal representa el estado objetivo que queremos alcanzar.
    def __init__(self, initial, goal=None):
        self.initial, self.goal = initial, goal

    # Devuelve las acciones que se pueden realizar desde un estado.
    # Aquí no se implementa porque cada problema tendrá acciones diferentes.
    def actions(self, state):
        raise NotImplementedError

    # Devuelve el nuevo estado que se obtiene
    # después de realizar una acción.
    # También debe ser implementado por cada problema específico.
    def result(self, state, action):
        raise NotImplementedError

    # Comprueba si el estado actual es el estado objetivo.
    def is_goal(self, state):
        return state == self.goal

    # Devuelve el costo de realizar una acción.
    # Por defecto, cada acción tiene costo 1.
    def step_cost(self, state, action):
        return 1


# La clase Node representa un nodo dentro del árbol o grafo de búsqueda.
class Node:

    # Limita los atributos que puede tener cada objeto Node.
    # Esto ayuda a ahorrar memoria cuando se crean muchos nodos.
    __slots__ = ("state", "parent", "action", "g")

    # Constructor de un nodo.
    # state = estado que representa el nodo.
    # parent = nodo anterior desde el cual llegamos.
    # action = acción utilizada para llegar a este nodo.
    # g = costo acumulado desde el inicio hasta este nodo.
    def __init__(self, state, parent=None, action=None, g=0):
        self.state, self.parent, self.action, self.g = state, parent, action, g

    # Reconstruye la secuencia de acciones desde el nodo inicial
    # hasta el nodo actual.
    def path(self):

        # node comienza siendo el nodo actual.
        # out será la lista donde se guardarán las acciones.
        node, out = self, []

        # Recorre hacia atrás usando parent
        # hasta llegar al nodo inicial.
        while node.parent is not None:

            # Guarda la acción utilizada para llegar a este nodo.
            out.append(node.action)

            # Avanza hacia el nodo padre.
            node = node.parent

        # Como recorrimos el camino desde el final hacia el inicio,
        # invertimos la lista para obtener el orden correcto.
        return out[::-1]

    # Define cómo se mostrará un objeto Node al imprimirlo.
    def __repr__(self):

        # Muestra el estado del nodo y su costo acumulado g.
        return f"Node({self.state}, g={self.g})"

## 2 · La frontera — lo *único* que cambia entre BFS, DFS y UCS

Tres estructuras de datos: **cola FIFO, pila LIFO y cola de prioridad**. El mismo ciclo de búsqueda.

Esta es la idea central de toda la semana.

In [2]:
# Clase que administra la "frontera" de búsqueda.
# La frontera guarda los nodos que todavía están pendientes por explorar.
class Frontier:
    """FIFO -> BFS.  LIFO -> DFS.  Priority by g -> UCS."""

    # Constructor de la frontera.
    # kind indica qué tipo de estrategia de búsqueda se va a usar:
    # "fifo"     -> BFS
    # "lifo"     -> DFS
    # "priority" -> UCS
    def __init__(self, kind):

        # Guarda el tipo de frontera.
        self.kind = kind

        # Crea un contador incremental.
        # Se usa en la cola de prioridad para desempatar nodos
        # que tengan el mismo costo g.
        self.counter = itertools.count()

        # Si la estrategia es FIFO o LIFO,
        # se utiliza deque para almacenar los nodos.
        if kind in ("fifo", "lifo"):
            self.data = deque()

        # Si la estrategia es por prioridad,
        # se utiliza una lista que funcionará como heap.
        elif kind == "priority":
            self.data = []

        # Si se recibe un tipo de frontera no válido,
        # se genera un error.
        else:
            raise ValueError(kind)

    # Agrega un nodo a la frontera.
    def push(self, node):

        # Si la frontera es de prioridad,
        # el nodo se guarda tomando en cuenta su costo acumulado g.
        if self.kind == "priority":

            # heapq mantiene los elementos ordenados por prioridad.
            # node.g es el costo.
            # next(self.counter) evita problemas si dos nodos tienen el mismo costo.
            # node es el nodo que se quiere guardar.
            heapq.heappush(self.data, (node.g, next(self.counter), node))

        # Para FIFO y LIFO simplemente se agrega el nodo al final.
        else:
            self.data.append(node)

    # Extrae un nodo de la frontera.
    def pop(self):

        # En una frontera de prioridad,
        # se extrae el nodo con menor costo g.
        if self.kind == "priority":
            return heapq.heappop(self.data)[2]

        # En FIFO, se extrae el primer nodo que entró.
        # Este comportamiento corresponde a BFS.
        if self.kind == "fifo":
            return self.data.popleft()

        # Si no es priority ni fifo, entonces es lifo.
        # Se extrae el último nodo que entró.
        # Este comportamiento corresponde a DFS.
        return self.data.pop()

    # Permite usar len(frontier)
    # para saber cuántos nodos hay actualmente en la frontera.
    def __len__(self):
        return len(self.data)

## 3 · La búsqueda genérica

Dos detalles de esta función son **fundamentales para que todo funcione correctamente**. Ambos están señalados con anotaciones.

Si omites cualquiera de los dos, aparecerán como errores en el **Duelo 1**.

In [3]:
# Función general de búsqueda.
# Recibe:
# problem -> el problema que queremos resolver.
# frontier_kind -> el tipo de frontera que determina el algoritmo:
# "fifo" = BFS, "lifo" = DFS, "priority" = UCS.
def search(problem, frontier_kind):

    # Crea el primer nodo usando el estado inicial del problema.
    # Como es el nodo inicial, todavía no tiene padre ni acción anterior.
    start = Node(problem.initial)

    # Comprueba si el estado inicial ya es el objetivo.
    # Si ya llegamos al objetivo sin buscar nada más,
    # devuelve el nodo inicial y 0 expansiones.
    if problem.is_goal(start.state):
        return start, 0

    # Crea la frontera según el tipo de búsqueda indicado.
    frontier = Frontier(frontier_kind)

    # Agrega el nodo inicial a la frontera.
    frontier.push(start)

    # Conjunto que guarda los estados que ya fueron explorados.
    # Sirve para evitar volver a procesar los mismos estados.
    explored = set()

    # Contador de cuántos nodos se han expandido.
    expansions = 0                       # instrument from line one

    # Mientras todavía existan nodos pendientes en la frontera...
    while frontier:

        # Extrae el siguiente nodo según el tipo de frontera.
        #
        # BFS -> primero que entró.
        # DFS -> último que entró.
        # UCS -> nodo con menor costo g.
        node = frontier.pop()

        # Comprueba si el estado del nodo es el objetivo.
        # El goal test se hace cuando el nodo sale de la frontera
        # para ser expandido.
        if problem.is_goal(node.state):  # goal test on EXPANSION, not generation
            return node, expansions

        # Si este estado ya fue explorado anteriormente,
        # se ignora este nodo y continúa con el siguiente.
        if node.state in explored:
            continue

        # Marca el estado actual como explorado.
        explored.add(node.state)

        # Como vamos a expandir este nodo,
        # aumentamos el contador de expansiones.
        expansions += 1

        # Obtiene todas las acciones posibles
        # desde el estado actual.
        for action in problem.actions(node.state):

            # Calcula el estado hijo que resulta
            # de aplicar esa acción.
            child_state = problem.result(node.state, action)

            # Solo considera el estado hijo si todavía
            # no ha sido explorado.
            if child_state not in explored:

                # Obtiene el costo de ejecutar esta acción.
                cost = problem.step_cost(node.state, action)

                # Crea un nuevo nodo hijo y lo agrega a la frontera.
                #
                # child_state -> nuevo estado.
                # node -> nodo padre.
                # action -> acción usada para llegar al hijo.
                # node.g + cost -> costo acumulado desde el inicio.
                frontier.push(Node(child_state, node, action, node.g + cost))

    # Si la frontera queda vacía y nunca se encontró
    # el objetivo, significa que no se encontró solución.
    # Devuelve None y el número total de expansiones realizadas.
    return None, expansions

## 4 · ⚠️ Momento de aprendizaje 1: comprobar el objetivo al expandir, no al generar

El error *clásico*. Este es el grafo de tres nodos que lo deja en evidencia:


```
start --(10)--> A
start --(1)---> B --(1)--> A
```

El camino óptimo hacia A tiene un costo de **2**. Un algoritmo UCS que comprueba si se alcanzó el objetivo cuando un nodo es *generado* devuelve **10**, porque `inicio -> A` se genera primero.

Ejecútalo. Luego recuérdalo cuando escribas el **Duelo 1**.

In [4]:
# Clase de ejemplo muy pequeña para probar el algoritmo de búsqueda.
# Hereda de Problem, por lo que debe definir acciones, resultado y costo.
class TinyGraph(Problem):

    # Grafo con tres estados principales:
    # "start" puede ir a A con costo 10
    # o a B con costo 1.
    #
    # Desde B se puede llegar a A con costo 1.
    #
    # Por lo tanto:
    # start -> A cuesta 10
    # start -> B -> A cuesta 2
    #
    # La ruta óptima hacia A es pasando por B.
    EDGES = {"start": [("A", 10), ("B", 1)], "B": [("A", 1)], "A": []}

    # Devuelve las acciones posibles desde un estado.
    # En este ejemplo, cada acción es simplemente
    # el estado destino al que podemos movernos.
    def actions(self, state):
        return [dst for dst, _ in self.EDGES[state]]

    # Devuelve el estado resultante de realizar una acción.
    # Como la acción ya contiene el destino,
    # simplemente devuelve action.
    def result(self, state, action):
        return action

    # Devuelve el costo de moverse desde el estado actual
    # hacia el estado indicado por action.
    def step_cost(self, state, action):
        return dict(self.EDGES[state])[action]


# Crea el problema:
# estado inicial = "start"
# objetivo = "A"
p = TinyGraph("start", "A")


# Ejecuta nuestra función search usando una frontera de prioridad.
# Esto corresponde a Uniform Cost Search (UCS).
node, exp = search(p, "priority")

# Muestra la ruta encontrada y su costo.
# Lo correcto debería ser:
# start -> B -> A
# costo total = 2
print(f"CORRECT (test on expansion): path={node.path()}  cost={node.g}")


# Esta es una versión incorrecta de search.
# El propósito es demostrar qué ocurre
# si se comprueba el objetivo demasiado pronto.
def search_buggy(problem, frontier_kind):
    """Idéntico, salvo que la prueba de objetivo se realiza en el momento de la generación."""

    # Crea el nodo inicial.
    start = Node(problem.initial)

    # Crea la frontera según el tipo indicado.
    frontier = Frontier(frontier_kind)

    # Agrega el nodo inicial a la frontera.
    frontier.push(start)

    # Guarda los estados ya explorados.
    explored = set()

    # Continúa mientras existan nodos pendientes.
    while frontier:

        # Extrae el siguiente nodo de la frontera.
        node = frontier.pop()

        # Si ya fue explorado, lo ignora.
        if node.state in explored:
            continue

        # Marca el estado como explorado.
        explored.add(node.state)

        # Recorre todas las acciones posibles.
        for action in problem.actions(node.state):

            # Obtiene el nuevo estado.
            s2 = problem.result(node.state, action)

            # Crea el nodo hijo y calcula
            # su costo acumulado desde el inicio.
            child = Node(s2, node, action,
                         node.g + problem.step_cost(node.state, action))

            # ERROR:
            # Aquí se comprueba si el hijo es el objetivo
            # en el mismo momento en que se genera.
            #
            # En UCS esto puede devolver una solución
            # antes de saber si existe otra ruta más barata.
            if problem.is_goal(s2):          # <-- BUG: tests too early
                return child, 0

            # Si todavía no era el objetivo,
            # agrega el nodo hijo a la frontera.
            frontier.push(child)

    # Si no encuentra solución, devuelve None.
    return None, 0


# Ejecuta la versión incorrecta del algoritmo.
bad, _ = search_buggy(p, "priority")

# Muestra la ruta incorrecta encontrada.
# Esta versión puede escoger directamente:
# start -> A
# con costo 10,
# aunque exista la ruta más barata de costo 2.
print(f"BUGGY   (test on generation): path={bad.path()}  cost={bad.g}   <-- not optimal")


# Comprueba automáticamente que:
# la versión correcta encuentra costo 2
# y la versión incorrecta encuentra costo 10.
assert node.g == 2 and bad.g == 10

# Mensaje final explicando la conclusión del ejemplo.
print("\nAmbas afirmaciones son válidas. La optimalidad de UCS depende de DÓNDE se realice la prueba.")

CORRECT (test on expansion): path=['B', 'A']  cost=2
BUGGY   (test on generation): path=['A']  cost=10   <-- not optimal

Ambas afirmaciones son válidas. La optimalidad de UCS depende de DÓNDE se realice la prueba.


## 5 · El rompecabezas de 8 piezas

El estado se representa como una tupla de 9 elementos; `0` representa el espacio vacío.

Esto hace concreta la idea de la diapositiva de que *la formulación del problema es una decisión de diseño*: elegimos una tupla plana para que los estados puedan usarse como claves hash y compararse en una sola operación.

In [5]:
# Estado objetivo del rompecabezas de 8 piezas.
# El número 0 representa el espacio vacío.
#
# 1 2 3
# 4 5 6
# 7 8 _
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)


# Clase que representa el problema del 8-puzzle.
# Hereda de la clase Problem que definimos anteriormente.
class EightPuzzle(Problem):

    # MOVES indica a qué posiciones puede moverse el espacio vacío
    # dependiendo de dónde se encuentre actualmente.
    #
    # Las posiciones del tablero se numeran así:
    #
    # 0 1 2
    # 3 4 5
    # 6 7 8
    #
    # Por ejemplo:
    # si el espacio vacío está en la posición 0,
    # puede intercambiarse con las posiciones 1 o 3.
    MOVES = {0: (1, 3), 1: (0, 2, 4), 2: (1, 5),
             3: (0, 4, 6), 4: (1, 3, 5, 7), 5: (2, 4, 8),
             6: (3, 7), 7: (4, 6, 8), 8: (5, 7)}

    # Constructor del problema.
    # initial representa la configuración inicial del tablero.
    # goal representa la configuración que queremos alcanzar.
    def __init__(self, initial, goal=GOAL):

        # Llama al constructor de la clase padre Problem
        # para guardar el estado inicial y el objetivo.
        super().__init__(initial, goal)

    # Devuelve las acciones posibles desde un estado.
    def actions(self, state):

        # state.index(0) busca la posición actual del espacio vacío.
        #
        # Luego MOVES devuelve las posiciones
        # con las que se puede intercambiar.
        return self.MOVES[state.index(0)]

    # Devuelve el nuevo estado después de realizar una acción.
    def result(self, state, action):

        # Convierte la tupla state en una lista
        # porque las listas sí se pueden modificar.
        s = list(state)

        # Busca la posición del espacio vacío.
        b = state.index(0)

        # Intercambia el espacio vacío con la posición
        # indicada por la acción.
        s[b], s[action] = s[action], s[b]

        # Convierte nuevamente la lista en una tupla
        # y devuelve el nuevo estado del tablero.
        return tuple(s)


# Función que muestra visualmente un estado del rompecabezas.
def show(state):

    # Recorre las posiciones 0, 3 y 6,
    # que representan el inicio de cada fila.
    for r in range(0, 9, 3):

        # Imprime tres elementos por fila.
        # Si el valor es 0, muestra "_" para representar
        # visualmente el espacio vacío.
        print(" ".join("_" if v == 0 else str(v) for v in state[r:r + 3]))


# Depths below are VERIFIED by exhaustive BFS from the goal (see the appendix
# cell at the bottom). Do not trust hand-labelled puzzle depths -- including mine.

# Estado que está a una distancia óptima de 2 movimientos del objetivo.
D2  = (1, 2, 3, 4, 5, 6, 0, 7, 8)      # optimal 2

# Estado que necesita como mínimo 8 movimientos para llegar al objetivo.
D8  = (0, 4, 2, 5, 1, 3, 7, 8, 6)      # optimal 8

# Estado que necesita como mínimo 12 movimientos para llegar al objetivo.
D12 = (5, 4, 2, 7, 0, 3, 8, 1, 6)      # optimal 12

# Estado que necesita como mínimo 16 movimientos.
# Este estado será utilizado nuevamente en la semana 4.
D16 = (7, 5, 2, 4, 0, 3, 8, 1, 6)      # optimal 16  <- week 4 uses this one


# Muestra visualmente la configuración D12.
show(D12)

5 4 2
7 _ 3
8 1 6


## 6 · Profundización iterativa

La memoria de DFS con la completitud de BFS.

La idea que se quiere comprobar es que el trabajo repetido representa un **factor constante**, no uno exponencial, porque el nivel más profundo domina la suma total.

In [6]:
# Búsqueda con profundidad limitada.
# Explora el árbol solo hasta una profundidad máxima indicada por "limit".
# Devuelve:
# node -> nodo solución, si encuentra uno.
# expansions -> cantidad de nodos expandidos.
# hit_limit -> indica si la búsqueda se detuvo por alcanzar el límite.
def depth_limited(problem, limit):
    """Búsqueda en árbol hasta una profundidad fija. Devuelve (nodo, expansiones, límite_alcanzado)."""

    # Lista usada como contador de expansiones.
    # Se usa una lista para poder modificarla dentro de la función recurse().
    counter = [0]

    # Función recursiva que realiza la búsqueda.
    # node representa el nodo actual.
    # depth indica cuánta profundidad todavía podemos explorar.
    def recurse(node, depth):

        # Primero comprueba si el nodo actual es el objetivo.
        if problem.is_goal(node.state):
            return node, False

        # Si llegamos al límite de profundidad,
        # detenemos la exploración de esta rama.
        #
        # True indica que se alcanzó el límite.
        if depth == 0:
            return None, True

        # Como este nodo va a ser expandido,
        # aumentamos el contador de expansiones.
        counter[0] += 1

        # Esta variable permite recordar si alguna rama
        # fue detenida porque alcanzó el límite.
        cutoff = False

        # Recorre todas las acciones posibles
        # desde el estado actual.
        for action in problem.actions(node.state):

            # Calcula el estado que resulta
            # después de realizar la acción.
            s2 = problem.result(node.state, action)

            # Evita regresar inmediatamente al estado padre.
            # Esto previene ciclos simples de dos estados.
            # skip the immediate parent -- cheap 2-cycle avoidance, no memory cost
            if node.parent is not None and s2 == node.parent.state:
                continue

            # Crea el nodo hijo.
            # node será su padre.
            # action indica cómo llegamos al hijo.
            # node.g + 1 aumenta el costo o profundidad acumulada.
            child = Node(s2, node, action, node.g + 1)

            # Continúa la búsqueda recursivamente,
            # reduciendo en 1 la profundidad disponible.
            result, hit = recurse(child, depth - 1)

            # Si encontramos una solución,
            # la devolvemos inmediatamente.
            if result is not None:
                return result, False

            # Guarda si alguna de las ramas
            # alcanzó el límite de profundidad.
            cutoff = cutoff or hit

        # Si ninguna rama encontró solución,
        # devuelve None.
        #
        # cutoff indica si la búsqueda quedó incompleta
        # porque se alcanzó el límite.
        return None, cutoff

    # Inicia la búsqueda desde el estado inicial
    # con el límite de profundidad indicado.
    node, hit = recurse(Node(problem.initial), limit)

    # Devuelve:
    # nodo solución,
    # número de expansiones,
    # y si se alcanzó el límite.
    return node, counter[0], hit


# IDS = Iterative Deepening Search.
# Realiza varias búsquedas con profundidad limitada,
# aumentando progresivamente el límite.
def ids(problem, max_depth=40):

    # Guarda el total de expansiones realizadas
    # en todas las búsquedas.
    total = 0

    # Prueba límites de profundidad:
    # 0, 1, 2, 3, ... hasta max_depth.
    for limit in range(max_depth + 1):

        # Ejecuta una búsqueda limitada
        # con el límite actual.
        node, exp, hit = depth_limited(problem, limit)

        # Acumula las expansiones realizadas.
        total += exp

        # Si se encontró una solución,
        # devuelve el nodo y el total de expansiones.
        if node is not None:
            return node, total

        # Si no se alcanzó el límite y tampoco hubo solución,
        # significa que ya se exploró todo el espacio disponible.
        if not hit:
            return None, total      # exhausted the whole space, no solution

    # Si se llega hasta max_depth sin encontrar solución,
    # devuelve None y el total de expansiones realizadas.
    return None, total


# Crea un problema EightPuzzle usando D8
# como estado inicial y ejecuta IDS.
node, exp = ids(EightPuzzle(D8))

# Muestra:
# cantidad de movimientos de la solución
# y cantidad total de nodos expandidos.
print(f"IDS: {len(node.path())} moves, {exp:,} expansions")

IDS: 8 moves, 233 expansions


## 7 · La tabla comparativa: deja que los números hablen por sí solos

In [7]:
# Importa el módulo time para medir cuánto tarda
# cada algoritmo en ejecutarse.
import time


# Lista de instancias del 8-puzzle que se van a probar.
# Cada elemento contiene:
# nombre descriptivo de la profundidad
# y el estado inicial correspondiente.
instances = [("depth 2", D2), ("depth 8", D8), ("depth 12", D12)]


# Recorre cada una de las instancias.
# label guarda el nombre, por ejemplo "depth 2".
# start guarda el estado inicial del puzzle.
for label, start in instances:

    # Crea un problema EightPuzzle usando
    # el estado inicial correspondiente.
    p = EightPuzzle(start)

    # Imprime el nombre de la instancia actual.
    print(f"\n{label}")

    # Imprime los encabezados de la tabla.
    #
    # algorithm   -> algoritmo utilizado.
    # expansions  -> cantidad de nodos expandidos.
    # sol.length  -> longitud de la solución.
    # optimal?    -> indica si encontró una solución óptima.
    # time        -> tiempo que tardó en ejecutarse.
    print(f"  {'algorithm':<10}{'expansions':>13}{'sol.length':>12}{'optimal?':>10}{'time':>9}")

    # Imprime una línea separadora para la tabla.
    print("  " + "-" * 54)

    # Lista donde se guardarán los resultados
    # de cada algoritmo.
    rows = []

    # Ejecuta BFS, DFS y UCS.
    #
    # BFS usa frontera FIFO.
    # DFS usa frontera LIFO.
    # UCS usa frontera de prioridad.
    for name, kind in [("BFS", "fifo"), ("DFS", "lifo"), ("UCS", "priority")]:

        # Guarda el tiempo justo antes
        # de ejecutar el algoritmo.
        t0 = time.perf_counter()

        # Ejecuta la búsqueda usando
        # el tipo de frontera correspondiente.
        node, exp = search(p, kind)

        # Calcula cuánto tiempo tardó la búsqueda.
        dt = time.perf_counter() - t0

        # Guarda los resultados:
        # nombre del algoritmo,
        # número de expansiones,
        # longitud de la solución,
        # y tiempo de ejecución.
        rows.append((name, exp, len(node.path()), dt))


    # Guarda el tiempo antes de ejecutar IDS.
    t0 = time.perf_counter()

    # Ejecuta Iterative Deepening Search.
    node, exp = ids(p)

    # Guarda también los resultados de IDS.
    rows.append(("IDS", exp, len(node.path()), time.perf_counter() - t0))


    # Busca la solución con menor cantidad de movimientos
    # entre todos los algoritmos ejecutados.
    #
    # Esa longitud se usa como referencia
    # para determinar cuáles encontraron una solución óptima.
    best = min(r[2] for r in rows)


    # Recorre los resultados guardados
    # para imprimirlos en forma de tabla.
    for name, exp, n, dt in rows:

        # Si la longitud de la solución encontrada
        # es igual a la mejor longitud encontrada,
        # se considera óptima.
        ok = "yes" if n == best else "NO"

        # Imprime una fila con:
        # algoritmo,
        # expansiones,
        # longitud de solución,
        # si es óptima,
        # y tiempo de ejecución.
        print(f"  {name:<10}{exp:>13,}{n:>12}{ok:>10}{dt:>8.3f}s")


depth 2
  algorithm    expansions  sol.length  optimal?     time
  ------------------------------------------------------
  BFS                   6           2       yes   0.000s
  DFS                   2           2       yes   0.000s
  UCS                   6           2       yes   0.000s
  IDS                   4           2       yes   0.000s

depth 8
  algorithm    expansions  sol.length  optimal?     time
  ------------------------------------------------------
  BFS                 172           8       yes   0.001s
  DFS               1,052        1048        NO   0.002s
  UCS                 172           8       yes   0.000s
  IDS                 233           8       yes   0.001s

depth 12
  algorithm    expansions  sol.length  optimal?     time
  ------------------------------------------------------
  BFS               2,177          12       yes   0.008s
  DFS               6,000        5944        NO   0.021s
  UCS               2,177          12       yes   0.011s
  I

### Lee la tabla en voz alta

- **DFS no es óptimo, y la tabla lo demuestra de forma contundente.** En la instancia de profundidad 12 devuelve una solución de cientos de movimientos. *Encontró* una meta, pero encontró una solución muy mala. La optimalidad no es algo que puedas evaluar simplemente mirando el resultado.

- **BFS y UCS coinciden exactamente** aquí, porque cada paso tiene un costo de 1. En `TinyGraph` no coincidirían; esa diferencia es precisamente la razón por la que existe UCS.

- **IDS sigue de cerca a BFS** en número de expansiones, mientras utiliza memoria O(bd) en lugar de O(b^d). Calcula `IDS / BFS` más abajo y observa qué tan pequeño es realmente el costo adicional.

> **Instrumenta desde la primera línea.** `expansions` no es un adorno: es la única medida con la que estos algoritmos pueden compararse, y corresponde al eje 3 de la tabla de evaluación.

> **Nunca escribas una búsqueda sin un contador.**

In [8]:
p = EightPuzzle(D12)
_, bfs_exp = search(p, "fifo")
_, ids_exp = ids(p)
print(f"BFS expansions : {bfs_exp:,}")
print(f"IDS expansions : {ids_exp:,}")
print(f"overhead ratio : {ids_exp / bfs_exp:.2f}x")
print("""

La cifra que aparece en el libro de texto para la sobrecarga de IDS con respecto a BFS
es de aproximadamente un 11% para un factor de ramificación cercano a 10.

El factor de ramificación efectivo del rompecabezas de 8 piezas es menor
(~1.7 después de comprobar el nodo padre), por lo que tu proporción será diferente
e incluso puede favorecer a IDS, porque IDS realiza una búsqueda en ÁRBOL y vuelve
a visitar estados que BFS almacena.

La idea importante es el ORDEN DE MAGNITUD: un factor constante, no uno exponencial.

Conseguiste un gran ahorro de memoria a cambio de una pequeña cantidad, limitada,
de trabajo repetido.
""")

BFS expansions : 2,177
IDS expansions : 4,192
overhead ratio : 1.93x


La cifra que aparece en el libro de texto para la sobrecarga de IDS con respecto a BFS
es de aproximadamente un 11% para un factor de ramificación cercano a 10.

El factor de ramificación efectivo del rompecabezas de 8 piezas es menor
(~1.7 después de comprobar el nodo padre), por lo que tu proporción será diferente
e incluso puede favorecer a IDS, porque IDS realiza una búsqueda en ÁRBOL y vuelve
a visitar estados que BFS almacena.

La idea importante es el ORDEN DE MAGNITUD: un factor constante, no uno exponencial.

Conseguiste un gran ahorro de memoria a cambio de una pequeña cantidad, limitada,
de trabajo repetido.



## 8 · Apéndice — verifica las profundidades por ti mismo

Nunca confíes ciegamente en una instancia de rompecabezas etiquetada manualmente, ni siquiera en las que aparecen en este notebook.

Esta celda realiza una búsqueda BFS exhaustiva hacia atrás desde el objetivo e imprime la verdadera profundidad óptima de cada una de las instancias utilizadas anteriormente.

Toma unos pocos segundos y recorre los 181 440 estados alcanzables, lo cual también sirve como recordatorio de lo pequeño que es el rompecabezas de 8 piezas y de lo poco viable que sería resolver el rompecabezas de 15 piezas con estos métodos.

In [9]:
# Importa deque nuevamente, pero con el alias "_dq".
# Se usa como una cola FIFO para realizar una búsqueda BFS.
from collections import deque as _dq


# Función que calcula la distancia óptima real
# desde el estado objetivo hacia todos los estados alcanzables.
def true_depths():

    # Diccionario que guarda:
    # estado -> distancia mínima al objetivo.
    #
    # El estado GOAL está a distancia 0 de sí mismo.
    dist = {GOAL: 0}

    # Crea una cola BFS y coloca inicialmente
    # el estado objetivo dentro de ella.
    q = _dq([GOAL])

    # Crea un EightPuzzle usando GOAL como estado inicial.
    # Se usa para poder acceder a actions() y result().
    pz = EightPuzzle(GOAL)

    # Mientras existan estados pendientes en la cola...
    while q:

        # Extrae el estado que lleva más tiempo en la cola.
        # Este comportamiento FIFO es propio de BFS.
        s = q.popleft()

        # Obtiene todas las acciones posibles
        # desde el estado actual.
        for a in pz.actions(s):

            # Calcula el nuevo estado que resulta
            # de ejecutar esa acción.
            t = pz.result(s, a)

            # Si este estado todavía no ha sido visitado...
            if t not in dist:

                # Su distancia será la distancia del estado anterior + 1.
                dist[t] = dist[s] + 1

                # Agrega el nuevo estado a la cola
                # para explorarlo posteriormente.
                q.append(t)

    # Devuelve el diccionario con todos los estados alcanzables
    # y su distancia óptima al objetivo.
    return dist


# Calcula todas las distancias óptimas.
dist = true_depths()


# Muestra:
# cuántos estados diferentes son alcanzables
# y cuál es la mayor distancia encontrada.
print(f"reachable states: {len(dist):,}   hardest instance: {max(dist.values())} moves\n")


# Recorre los cuatro estados utilizados anteriormente:
# D2, D8, D12 y D16.
for name, s in [("D2", D2), ("D8", D8), ("D12", D12), ("D16", D16)]:

    # Muestra el nombre del estado,
    # su configuración completa
    # y su profundidad óptima real.
    print(f"  {name:<4} {s}  true optimal depth = {dist[s]}")


# Comprueba automáticamente que las etiquetas
# D2, D8, D12 y D16 realmente corresponden
# a profundidades óptimas de 2, 8, 12 y 16.
#
# Si alguna condición fuera falsa,
# Python produciría un AssertionError.
assert dist[D2] == 2 and dist[D8] == 8 and dist[D12] == 12 and dist[D16] == 16


# Si el assert anterior se cumple,
# muestra un mensaje confirmando que las cuatro etiquetas son correctas.
print("\nAll four labels verified.")

reachable states: 181,440   hardest instance: 31 moves

  D2   (1, 2, 3, 4, 5, 6, 0, 7, 8)  true optimal depth = 2
  D8   (0, 4, 2, 5, 1, 3, 7, 8, 6)  true optimal depth = 8
  D12  (5, 4, 2, 7, 0, 3, 8, 1, 6)  true optimal depth = 12
  D16  (7, 5, 2, 4, 0, 3, 8, 1, 6)  true optimal depth = 16

All four labels verified.


## 9 · Guarda esto para más adelante

La Semana 4 importa `Problem`, `Node`, `search`, `EightPuzzle`, `GOAL` y `D16`, y añade **diez líneas** para obtener A\*.

La Semana 9 reutiliza `search` sin modificaciones para un planificador STRIPS.

Copia ahora las clases de este notebook en `aicourse/search.py` dentro de tu repositorio, ya que tanto el studio como el Duelo 1 esperan encontrarlas allí.